# 05: Minimum wage extension

Author's specification (`refs/author_spec.md`, "Minimum wage extension"). Base covariates and samples as in the base model.
Each occupation is estimated for 1983–2025 and 2001–2025 separately.

Inputs: `data/minimumwage_annual.csv` (FRED state minimum wage series, read here only), `outputs/py/dk_clean.parquet`.
Outputs: `outputs/py/minwage_results.csv`, `outputs/py/minwage_mechanism.csv`, `outputs/figures/figure3_minwage_mechanism_2001_2025.png`,
`outputs/RESULTS_minwage.md`.

Paths can be overridden with the environment variables `MW_PATH`, `DK_PATH`, and `MW_OUT` (used for test runs on
synthetic inputs; defaults are the project paths).

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pyfixest as pf
from scipy import stats

OUT = os.environ.get("MW_OUT", "../outputs")
MW_PATH = os.environ.get("MW_PATH", "../data/minimumwage_annual.csv")
DK_PATH = os.environ.get("DK_PATH", "../outputs/py/dk_clean.parquet")
os.makedirs(f"{OUT}/py", exist_ok=True)
os.makedirs(f"{OUT}/figures", exist_ok=True)
plt.rcParams["font.family"] = ["Times New Roman", "Times", "serif"]

AUTHOR_COV = ["union", "parttime", "educ_cat", "female", "black", "hispanic", "age", "age2",
              "msa", "central_city", "cc_unknown", "geo_unidentified"]
ANALYSIS_COLS = ["occupation", "YEAR", "STATEFIP", "outsourced", "EARNWT", "ym", "hourly_wage", "log_wage",
                 "extension_sample"] + AUTHOR_COV
dk = pd.read_parquet(DK_PATH, columns=ANALYSIS_COLS)
print("analysis rows:", dk.shape)

analysis rows: (166391, 21)


## Crosswalk and state-year grid

Postal codes map to STATEFIP through the standard FIPS codes. The grid covers every STATEFIP in the analysis data and
every year 1983–2025, so states without a minimum-wage column are included.

In [2]:
FIPS = {"AL":1,"AK":2,"AZ":4,"AR":5,"CA":6,"CO":8,"CT":9,"DE":10,"DC":11,"FL":12,"GA":13,"HI":15,"ID":16,
        "IL":17,"IN":18,"IA":19,"KS":20,"KY":21,"LA":22,"ME":23,"MD":24,"MA":25,"MI":26,"MN":27,"MS":28,
        "MO":29,"MT":30,"NE":31,"NV":32,"NH":33,"NJ":34,"NM":35,"NY":36,"NC":37,"ND":38,"OH":39,"OK":40,
        "OR":41,"PA":42,"RI":44,"SC":45,"SD":46,"TN":47,"TX":48,"UT":49,"VT":50,"VA":51,"WA":53,"WV":54,
        "WI":55,"WY":56}
analysis_states = sorted(dk["STATEFIP"].unique())
assert set(analysis_states) <= set(FIPS.values()), set(analysis_states) - set(FIPS.values())
print("analysis states:", len(analysis_states))

analysis states: 51


In [3]:
mw_raw = pd.read_csv(MW_PATH)
mw_raw["YEAR"] = mw_raw["observation_date"].astype(str).str[:4].astype(int)
assert mw_raw["YEAR"].is_unique, "expected one row per year"

DROP = {"STTMINWGFG", "STTMINWGGU", "STTMINWGPR"}
state_cols = [c for c in mw_raw.columns if c.startswith("STTMINWG") and c not in DROP]
postal = [c[len("STTMINWG"):] for c in state_cols]
unknown = set(postal) - set(FIPS)
assert not unknown, f"postal codes without FIPS: {unknown}"
print("state columns:", len(state_cols), "| states without a column:",
      sorted(set(FIPS) - set(postal)))

long = mw_raw.melt(id_vars=["YEAR"], value_vars=state_cols, var_name="col", value_name="state_rate")
long["STATEFIP"] = long["col"].str[len("STTMINWG"):].map(FIPS)
long = long[["STATEFIP", "YEAR", "state_rate"]]
fed = mw_raw[["YEAR", "STTMINWGFG"]].rename(columns={"STTMINWGFG": "fed_rate"})

GRID_YEARS = pd.DataFrame({"YEAR": range(1983, 2026)})
GRID_STATES = pd.DataFrame({"STATEFIP": analysis_states})
grid = (GRID_STATES.merge(GRID_YEARS, how="cross")
        .merge(long, on=["STATEFIP", "YEAR"], how="left", validate="1:1")
        .merge(fed, on="YEAR", how="left", validate="m:1"))
assert grid["fed_rate"].notna().all(), "federal minimum missing for some grid year"

grid["binding_mw"] = np.fmax(grid["state_rate"], grid["fed_rate"])  # NaN state rate -> federal rate
grid["state_above_fed"] = grid["state_rate"] > grid["fed_rate"]     # NaN compares False
grid = grid.sort_values(["STATEFIP", "YEAR"]).reset_index(drop=True)
grid["log_mw"] = np.log(grid["binding_mw"])
grid["log_mw_lead1"] = grid.groupby("STATEFIP")["log_mw"].shift(-1)
grid["log_mw_lead2"] = grid.groupby("STATEFIP")["log_mw"].shift(-2)
assert grid["binding_mw"].notna().all()

decade = (grid["YEAR"] // 10 * 10).astype(str) + "s"
count_by_decade = (grid.assign(decade=decade)
                   .groupby("decade")
                   .agg(state_years=("state_above_fed", "size"),
                        state_above_federal=("state_above_fed", "sum"))
                   .reset_index())
print(count_by_decade.to_string(index=False))

state columns: 46 | states without a column: ['AL', 'LA', 'MS', 'SC', 'TN']
decade  state_years  state_above_federal
 1980s          357                   35
 1990s          510                   86
 2000s          510                  182
 2010s          510                  246
 2020s          306                  185


## Merge onto the analysis data

Many-to-one merge by STATEFIP and YEAR. No missing binding minimum is allowed after the merge.

In [4]:
dk_mw = dk.merge(grid[["STATEFIP", "YEAR", "binding_mw", "log_mw", "log_mw_lead1", "log_mw_lead2"]],
                 on=["STATEFIP", "YEAR"], how="left", validate="m:1")
assert len(dk_mw) == len(dk), "merge changed row count"
assert dk_mw["binding_mw"].notna().all(), "missing binding_mw after merge"
print("merged rows:", len(dk_mw), "| missing binding_mw:", int(dk_mw["binding_mw"].isna().sum()))

dk_mw["os_x_logmw"] = dk_mw["outsourced"] * dk_mw["log_mw"]
dk_mw["os_x_lead1"] = dk_mw["outsourced"] * dk_mw["log_mw_lead1"]
dk_mw["os_x_lead2"] = dk_mw["outsourced"] * dk_mw["log_mw_lead2"]
dk_mw["os_x_yearc"] = dk_mw["outsourced"] * (dk_mw["YEAR"] - 2001)
dk_mw["os_state"] = dk_mw["outsourced"].astype(str) + "_" + dk_mw["STATEFIP"].astype(str)
dk_mw["os_year"] = dk_mw["outsourced"].astype(str) + "_" + dk_mw["YEAR"].astype(str)
print("bite: skipped (the extract contains only janitors and guards, not all workers in each state-year)")

merged rows: 166391 | missing binding_mw: 0
bite: skipped (the extract contains only janitors and guards, not all workers in each state-year)


## 1. Mechanism: share earning below 1.1 x binding minimum

Weighted (EARNWT > 0) share of base-sample workers with hourly_wage < 1.1 x binding_mw, by outsourcing status and period.
Same-year comparison, so the CPI deflator cancels. Bins: the 13 by-period bins for 1983–2025, and the 7 bins from 2001 on for 2001–2025.

In [ ]:
BINS = [(1983,1985),(1986,1988),(1989,1991),(1992,1994),(1995,1997),(1998,2000),(2001,2003),(2004,2006),
        (2007,2009),(2010,2012),(2013,2015),(2016,2019),(2021,2025)]
SAMPLES = {"1983-2025": (1983, 2025), "2001-2025": (2001, 2025)}
OCC_LABEL = {"janitor": "Janitors", "guard": "Guards"}

mech_rows = []
for occ in ["janitor", "guard"]:
    for samp, (s0, s1) in SAMPLES.items():
        for (y0, y1) in BINS:
            if y0 < s0 or y1 > s1:
                continue
            g = dk_mw[(dk_mw["occupation"] == occ) & dk_mw["YEAR"].between(y0, y1)
                      & dk_mw["hourly_wage"].notna() & (dk_mw["EARNWT"] > 0)]
            for status in (0, 1):
                h = g[g["outsourced"] == status]
                below = (h["hourly_wage"] < 1.1 * h["binding_mw"]).astype(float)
                share = np.average(below, weights=h["EARNWT"])
                mech_rows.append(dict(occupation=OCC_LABEL[occ], sample=samp, period=f"{y0}-{y1}",
                                      outsourced=status, share_below_1p1_mw=share, n_obs=len(h)))
mechanism = pd.DataFrame(mech_rows)
mechanism.to_csv(f"{OUT}/py/minwage_mechanism.csv", index=False)
print(mechanism.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, occ in zip(axes, ["Janitors", "Guards"]):
    for status, color, ls, lab in [(0, "#7a7a7a", "--", "In-house"), (1, "#000000", "-", "Outsourced")]:
        sub = mechanism[(mechanism["occupation"] == occ) & (mechanism["sample"] == "2001-2025") & (mechanism["outsourced"] == status)]
        ax.plot(sub["period"], sub["share_below_1p1_mw"], marker="o", linestyle=ls, color=color, label=lab)
    ax.set_title(occ)
    ax.tick_params(axis="x", rotation=45)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
axes[0].set_ylabel("Share earning below 1.1 x binding minimum")
axes[0].legend(title="Outsourcing status")
fig.suptitle("Share Earning Below 1.1 x Binding Minimum by Period, 2001-2025")
fig.tight_layout()
for ax in axes:
    ax.set_ylim(bottom=0)
fig.savefig(f"{OUT}/figures/figure3_minwage_mechanism_2001_2025.png", dpi=300)
plt.show()

## 2–4. Models

For each occupation and sample: samples are restricted to rows with non-missing log_wage and every covariate, and log_mw.
Clustering: survey month (`ym`) by default; state clusters are reported as robustness for the interaction and lead terms.
`outsourced^STATEFIP` and `outsourced^YEAR` are coded as the string categories `os_state` and `os_year`.

In [ ]:
COVS = " + ".join(AUTHOR_COV)
F_INT = f"log_wage ~ os_x_logmw + {COVS} | STATEFIP^YEAR + os_state + os_year"
F_LEAD = f"log_wage ~ os_x_logmw + os_x_lead1 + os_x_lead2 + {COVS} | STATEFIP^YEAR + os_state + os_year"
F_TR_NO = f"log_wage ~ os_x_yearc + {COVS} | STATEFIP^YEAR + os_state"
F_TR_MW = f"log_wage ~ os_x_yearc + os_x_logmw + {COVS} | STATEFIP^YEAR + os_state"

def fit(formula, data, cluster="ym"):
    return pf.feols(formula, data=data, weights="EARNWT", vcov={"CRV1": cluster})

def wald_zero(f, names):
    coef_index = list(f.coef().index)
    R = np.zeros((len(names), len(coef_index)))
    for i, n in enumerate(names):
        R[i, coef_index.index(n)] = 1
    w = f.wald_test(R=R, distribution="chi2")
    return float(w["statistic"]), float(w["pvalue"]), len(names)

res_rows = []
def record(occ, samp, model, term, f, fstate, n, t=None, p=None, stat=None, df=None, scale=1.0):
    if term is not None:
        r = f.tidy().loc[term]
        rs = fstate.tidy().loc[term] if fstate is not None else None
        res_rows.append(dict(occ=OCC_LABEL[occ], sample=samp, model=model, term=term,
                             estimate=scale * r["Estimate"], se=scale * r["Std. Error"], se_state=(rs["Std. Error"] if rs is not None else np.nan),
                             t=r["t value"], p=r["Pr(>|t|)"], p_state=(rs["Pr(>|t|)"] if rs is not None else np.nan),
                             stat=np.nan, df=np.nan, n_obs=n))
    else:
        res_rows.append(dict(occ=OCC_LABEL[occ], sample=samp, model=model, term=None,
                             estimate=np.nan, se=np.nan, se_state=np.nan, t=np.nan, p=p,
                             p_state=np.nan, stat=stat, df=df, n_obs=n))

for occ in ["janitor", "guard"]:
    for samp, (s0, s1) in SAMPLES.items():
        base = dk_mw[(dk_mw["occupation"] == occ) & dk_mw["YEAR"].between(s0, s1)]
        s = base.dropna(subset=["log_wage", "log_mw"] + AUTHOR_COV).copy()
        s_lead = s.dropna(subset=["log_mw_lead1", "log_mw_lead2"]).copy()

        # 2. interaction model
        f2 = fit(F_INT, s)
        f2s = fit(F_INT, s, cluster="STATEFIP")
        record(occ, samp, "interaction", "os_x_logmw", f2, f2s, f2._N)

        # 3. leads: all three coefficients, joint Wald on leads (lead sample), and model 2 on the same sample
        f2_lead_sample = fit(F_INT, s_lead)
        f2_lead_state = fit(F_INT, s_lead, cluster="STATEFIP")
        record(occ, samp, "interaction_leadsample", "os_x_logmw", f2_lead_sample, f2_lead_state, f2_lead_sample._N)
        f3 = fit(F_LEAD, s_lead)
        f3s = fit(F_LEAD, s_lead, cluster="STATEFIP")
        for term in ["os_x_logmw", "os_x_lead1", "os_x_lead2"]:
            record(occ, samp, "leads", term, f3, f3s, f3._N)
        stat, pv, dfw = wald_zero(f3, ["os_x_lead1", "os_x_lead2"])
        record(occ, samp, "leads", None, None, None, f3._N, p=pv, stat=stat, df=dfw)

        # 4. trend decomposition, same sample and FE
        f4a = fit(F_TR_NO, s)
        f4b = fit(F_TR_MW, s)
        record(occ, samp, "trend_without_mw", "os_x_yearc", f4a, None, f4a._N, scale=10)
        record(occ, samp, "trend_with_mw", "os_x_yearc", f4b, None, f4b._N, scale=10)

minwage_results = pd.DataFrame(res_rows)
minwage_results.to_csv(f"{OUT}/py/minwage_results.csv", index=False)
print(minwage_results.round(4).to_string(index=False))

## Write RESULTS_minwage.md

Descriptive summary of `outputs/py/minwage_results.csv` and `outputs/py/minwage_mechanism.csv`. Associations, not causal effects.

In [ ]:
rw = minwage_results
def g(occ, samp, model, term=None):
    m = rw[(rw["occ"] == occ) & (rw["sample"] == samp) & (rw["model"] == model)]
    m = m[m["term"].isna()] if term is None else m[m["term"] == term]
    assert len(m) == 1, (occ, samp, model, term, len(m))
    return m.iloc[0]

def pfmt(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

def sgn(v, d=3):
    return f"{v:+.{d}f}"

def mech_table(samp):
    sub = (mechanism[mechanism["sample"] == samp]
           .assign(share=lambda d: d["share_below_1p1_mw"].map(lambda v: f"{100*v:.1f}%"))
           .pivot_table(index=["occupation", "period"], columns="outsourced", values="share", aggfunc="first")
           .reset_index())
    sub.columns = ["occupation", "period", "in_house", "outsourced"]
    out = ["| Occupation | Period | In-house | Outsourced |", "|---|---|---|---|"]
    for _, r in sub.iterrows():
        out.append(f"| {r['occupation']} | {r['period']} | {r['in_house']} | {r['outsourced']} |")
    return out

def interaction_block(samp):
    out = []
    for occ in ["Janitors", "Guards"]:
        i_ = g(occ, samp, "interaction", "os_x_logmw")
        ls = g(occ, samp, "interaction_leadsample", "os_x_logmw")
        w = g(occ, samp, "leads", None)
        tn = g(occ, samp, "trend_without_mw", "os_x_yearc")
        tw = g(occ, samp, "trend_with_mw", "os_x_yearc")
        out.append(f"- {occ}, {samp}: outsourced x log minimum wage {sgn(i_.estimate)} (SE {i_.se:.3f}, {pfmt(i_.p)}; "
                   f"state-clustered SE {i_.se_state:.3f}, {pfmt(i_.p_state)}).")
        out.append(f"  - Lead sample (n = {int(ls.n_obs):,}): interaction {sgn(ls.estimate)} (SE {ls.se:.3f}). "
                   f"Joint test that the leads are zero: chi2 = {w.stat:.2f}, df = {int(w.df)}, {pfmt(w.p)}.")
        out.append(f"  - Trend per decade, without log_mw {sgn(tn.estimate)} (SE {tn.se:.3f}, {pfmt(tn.p)}); "
                   f"with log_mw {sgn(tw.estimate)} (SE {tw.se:.3f}, {pfmt(tw.p)}).")
    return out

lines = ["# Results: minimum wage extension", "",
         "Regenerated by `notebooks/05_minwage.ipynb` from `outputs/py/`. Associations, not causal effects. "
         "Body is 2001-2025 (per CLAUDE.md's Paper structure); the appendix holds the 1983-2025 results. "
         "Clustering: survey month (ym); state-clustered SEs are shown in `minwage_results.csv` as robustness.", "",
         "## Body (2001-2025)", "",
         "### Mechanism: share earning below 1.1 x binding minimum", "",
         "Figure: `outputs/figures/figure3_minwage_mechanism_2001_2025.png` (body Figure 3).", ""]
lines += mech_table("2001-2025")
lines += ["", "### Interaction, leads, and trends", ""] + interaction_block("2001-2025")
lines += ["", "## Appendix (1983-2025)", "",
          "### Mechanism: share earning below 1.1 x binding minimum", ""]
lines += mech_table("1983-2025")
lines += ["", "### Interaction, leads, and trends", ""] + interaction_block("1983-2025")
lines += ["", "## Data notes", "",
          "- Binding minimum = max(state, federal); missing state rate uses the federal rate. Alabama, Louisiana, "
          "Mississippi, South Carolina, and Tennessee have no state column and use the federal rate.",
          "- Bite is not computed: the extract contains only janitors and guards, not all workers in each state-year.",
          "- The lead models use the lead sample; the interaction model is re-estimated on the same rows for comparison.",
          "- Trend coefficients are per year in the model; they are reported here per decade (x10).",
          ""]
with open(f"{OUT}/RESULTS_minwage.md", "w") as f:
    f.write("\n".join(lines))
print("\n".join(lines))